In [1]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import math
import os
import itertools
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


Device: cpu


In [2]:
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/8"

train_path = os.path.join(SAMPLED_DATA_DIR, "hm_subset_train.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, "hm_subset_test.csv")

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

# Cast IDs to str to handle mixed-type H&M IDs consistently
for df in [train_df, test_df]:
    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)

# Build contiguous 0-based encoders fitted on train only
user_enc = {u: i for i, u in enumerate(sorted(train_df["user_id"].unique()))}
item_enc = {v: i for i, v in enumerate(sorted(train_df["item_id"].unique()))}

def remap(df, user_enc, item_enc, drop_unknown=False):
    df = df.copy()
    if drop_unknown:
        df = df[df["user_id"].isin(user_enc) & df["item_id"].isin(item_enc)]
    df["user_id"] = df["user_id"].map(user_enc)
    df["item_id"] = df["item_id"].map(item_enc)
    return df.dropna(subset=["user_id", "item_id"]).astype({"user_id": int, "item_id": int})

train_df = remap(train_df, user_enc, item_enc)
test_df  = remap(test_df,  user_enc, item_enc, drop_unknown=True)

train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=0)

# Dimensions from encoder (not from split subsets)
n_users = len(user_enc)
n_items = len(item_enc)
print(f"Total Users : {n_users}")
print(f"Total Items : {n_items}")
train_df.head()


Total Users : 1760
Total Items : 2881


,user_id,item_id,bought
17812,23,2123,1
788,491,714,1
1399,357,1741,1
12060,590,1828,1
31118,1248,1025,1


In [3]:
min_rating = float(train_df["bought"].min())
max_rating = float(train_df["bought"].max())

def make_matrix(df, n_u, n_i, min_r, max_r):
    mat   = torch.full((n_u, n_i), -1.0)
    users = torch.tensor(df["user_id"].values, dtype=torch.long)
    items = torch.tensor(df["item_id"].values, dtype=torch.long)
    vals  = torch.tensor((df["bought"].values - min_r) / max(max_r - min_r, 1e-8),
                         dtype=torch.float32)
    mask  = (users >= 0) & (users < n_u) & (items >= 0) & (items < n_i)
    if (~mask).any():
        print(f"  [make_matrix] dropping {(~mask).sum().item()} out-of-bounds rows")
    mat[users[mask], items[mask]] = vals[mask]
    return mat

rating_matrix      = make_matrix(train_df, n_users, n_items, min_rating, max_rating)
val_rating_matrix  = make_matrix(val_df,   n_users, n_items, min_rating, max_rating)
test_rating_matrix = make_matrix(test_df,  n_users, n_items, min_rating, max_rating)

print(f"Train obs : {(rating_matrix      != -1).sum().item()}")
print(f"Val obs   : {(val_rating_matrix  != -1).sum().item()}")
print(f"Test obs  : {(test_rating_matrix != -1).sum().item()}")
print(f"Matrix shape: {rating_matrix.shape}")


Train obs : 61386
Val obs   : 15572
Test obs  : 19450
Matrix shape: torch.Size([1760, 2881])


In [4]:
# Alias so the rest of the FCF cells work unchanged
n_movies = n_items
print(f"n_users={n_users}  n_movies={n_movies}")


n_users=1760  n_movies=2881


In [5]:
# Rating matrix already built above with -1 for missing entries.
# Verify a few values:
obs = (rating_matrix != -1).float().sum().item()
print(f"Observed train entries: {int(obs)}")


Observed train entries: 61386


In [6]:
# # This is how we can define our feature matrices
# # We're going to be training these, so we'll need gradients
# latent_vectors = 5
# user_features = torch.randn(n_users, latent_vectors, requires_grad=True)
# user_features.data.mul_(0.01)
# movie_features = torch.randn(n_movies, latent_vectors, requires_grad=True)
# movie_features.data.mul_(0.01)

In [7]:
class FCFLoss(torch.nn.Module):
    def __init__(self, lam_u=0.3, lam_v=0.3):
        super().__init__()
        self.lam_u = lam_u
        self.lam_v = lam_v
    
    def forward(self, matrix, u_features, v_features):
        non_zero_mask = (matrix != -1).type(torch.FloatTensor)
        predicted = torch.sigmoid(torch.mm(u_features, v_features.t()))
        
        diff = (matrix - predicted)**2
        prediction_error = torch.sum(diff*non_zero_mask)

        u_regularization = self.lam_u * torch.sum(u_features.norm(dim=1))
        v_regularization = self.lam_v * torch.sum(v_features.norm(dim=1))
        
        return prediction_error + u_regularization + v_regularization, prediction_error

## 4. Parameter Tuning — Grid Search

Grid over `lr`, `lam` (regularisation), and `latent_vectors`.
Uses the **validation matrix** for early stopping; writes `lr`, `lam`, `latent_vectors`.


In [9]:
# ── Grid ─────────────────────────────────────────────────────────────────────
LR_GRID      = [0.001, 0.01,  0.05]
LAM_GRID     = [0.01,  0.1,   0.3 ]
LATENT_GRID  = [10,    20,    40  ]
TUNE_EPOCHS  = 30
TUNE_PAT     = 5          # early-stop patience on val RMSE
TUNE_CLIENTS = min(n_users, 200)
TUNE_M       = n_users // TUNE_CLIENTS

param_grid = list(itertools.product(LR_GRID, LAM_GRID, LATENT_GRID))
print(f"Grid search: {len(param_grid)} combinations x up to {TUNE_EPOCHS} epochs")
print(f"{'#':>3}  {'lr':>7}  {'lam':>6}  {'K':>4}  {'best val RMSE':>14}")
print("-" * 42)

grid_results = []

for _k, (_lr, _lam, _K) in enumerate(param_grid, 1):
    torch.manual_seed(0)

    _uf  = [torch.randn(TUNE_M, _K, requires_grad=True) for _ in range(TUNE_CLIENTS)]
    for u in _uf: u.data.mul_(0.01)
    _vf  = torch.randn(n_movies, _K, requires_grad=True)
    _vf.data.mul_(0.01)

    _loss_fn   = FCFLoss(lam_u=_lam, lam_v=_lam)
    _opt_cli   = [torch.optim.Adam([u], lr=_lr) for u in _uf]
    _opt_srv   = torch.optim.Adam([_vf], lr=_lr)

    _best_val, _wait = float('inf'), 0

    for _ep in range(TUNE_EPOCHS):
        _opt_srv.zero_grad()
        for i in range(TUNE_CLIENTS):
            _opt_cli[i].zero_grad()
            _loss, _ = _loss_fn(
                rating_matrix[:TUNE_CLIENTS * TUNE_M][i*TUNE_M:(i+1)*TUNE_M],
                _uf[i], _vf)
            _loss.backward()
            _opt_cli[i].step()
        _opt_srv.step()

        # Val RMSE
        with torch.no_grad():
            _val_mask = (val_rating_matrix[:TUNE_CLIENTS * TUNE_M] != -1).float()
            _n_val    = _val_mask.sum()
            if _n_val == 0:
                _val_rmse = float('inf')
            else:
                _se = 0.0
                for i in range(TUNE_CLIENTS):
                    _pred = torch.sigmoid(torch.mm(_uf[i], _vf.t()))
                    _pred_sc = (_pred * (max_rating - min_rating) + min_rating) * _val_mask[i*TUNE_M:(i+1)*TUNE_M]
                    _true_sc = val_rating_matrix[i*TUNE_M:(i+1)*TUNE_M] * _val_mask[i*TUNE_M:(i+1)*TUNE_M]
                    _se += ((_pred_sc - _true_sc) ** 2).sum().item()
                _val_rmse = (_se / _n_val.item()) ** 0.5

        if _val_rmse < _best_val:
            _best_val, _wait = _val_rmse, 0
        else:
            _wait += 1
            if _wait >= TUNE_PAT:
                break

    grid_results.append((_best_val, _lr, _lam, _K))
    _marker = ' <--' if _best_val == min(r[0] for r in grid_results) else ''
    print(f"{_k:>3}  {_lr:>7.4f}  {_lam:>6.3f}  {_K:>4d}  {_best_val:>14.4f}{_marker}")

_best = min(grid_results, key=lambda x: x[0])
best_val_rmse_tune, best_lr, best_lam, best_K = _best
print(f"\nBest val RMSE  : {best_val_rmse_tune:.4f}")
print(f"  lr             = {best_lr}")
print(f"  lam            = {best_lam}")
print(f"  latent_vectors = {best_K}")
lr             = best_lr
lam            = best_lam
latent_vectors = best_K
print("\nHyperparameters set. Run the training cell.")


Grid search: 27 combinations x up to 30 epochs
  #       lr     lam     K   best val RMSE
------------------------------------------
  1   0.0010   0.010    10         12.4800 <--
  2   0.0010   0.010    20         12.4800 <--
  3   0.0010   0.010    40         12.4795 <--
  4   0.0010   0.100    10         12.4800
  5   0.0010   0.100    20         12.4800
  6   0.0010   0.100    40         12.4800
  7   0.0010   0.300    10         12.4801
  8   0.0010   0.300    20         12.4800
  9   0.0010   0.300    40         12.4800
 10   0.0100   0.010    10         12.0827 <--
 11   0.0100   0.010    20         10.4028 <--
 12   0.0100   0.010    40          7.8526 <--
 13   0.0100   0.100    10         12.4800
 14   0.0100   0.100    20         12.4800
 15   0.0100   0.100    40         12.4800
 16   0.0100   0.300    10         12.4801
 17   0.0100   0.300    20         12.4800
 18   0.0100   0.300    40         12.4800
 19   0.0500   0.010    10          3.1193 <--
 20   0.0500   0.010  

In [10]:
# ── Use tuned hyperparameters (fall back to defaults if grid search skipped) ─
try: latent_vectors
except NameError: latent_vectors = 20
try: lr
except NameError: lr = 0.01
try: lam
except NameError: lam = 0.01

num_client = min(n_users, 200)
m          = n_users // num_client
num_epoch  = 100
patience   = 10   # early stopping on val RMSE

rating_matrix_fcf = rating_matrix[:num_client * m]

torch.manual_seed(0)
user_features = [torch.randn(m, latent_vectors, requires_grad=True)
                 for _ in range(num_client)]
for uf in user_features: uf.data.mul_(0.01)

movie_features = torch.randn(n_movies, latent_vectors, requires_grad=True)
movie_features.data.mul_(0.01)

fcferror         = FCFLoss(lam_u=lam, lam_v=lam)
optimizer_client = [torch.optim.Adam([uf], lr=lr) for uf in user_features]
optimizer_server = torch.optim.Adam([movie_features], lr=lr)

error_list        = []
previous          = torch.zeros(n_movies, latent_vectors)
best_val_rmse     = float('inf')
patience_count    = 0
best_movie_state  = None
best_user_states  = None

val_mask = (val_rating_matrix[:num_client * m] != -1).float()
n_val    = val_mask.sum()

for step in range(num_epoch):
    optimizer_server.zero_grad()
    total_loss            = 0
    aver_prediction_error = 0

    for i in range(num_client):
        optimizer_client[i].zero_grad()
        loss, prediction_error = fcferror(
            rating_matrix_fcf[i*m:(i+1)*m], user_features[i], movie_features)
        aver_prediction_error += prediction_error / num_client
        total_loss            += loss
        loss.backward()
        optimizer_client[i].step()

    optimizer_server.step()

    # Convergence tracker
    error = torch.norm(movie_features - previous, p=2) / torch.norm(previous, p=2).clamp(min=1e-8)
    error_list.append(error.detach().numpy())
    previous = movie_features.clone()

    # Val RMSE for early stopping
    with torch.no_grad():
        se = 0.0
        for i in range(num_client):
            pred    = torch.sigmoid(torch.mm(user_features[i], movie_features.t()))
            pred_sc = (pred * (max_rating - min_rating) + min_rating) * val_mask[i*m:(i+1)*m]
            true_sc = val_rating_matrix[i*m:(i+1)*m] * val_mask[i*m:(i+1)*m]
            se     += ((pred_sc - true_sc) ** 2).sum().item()
        val_rmse = (se / n_val.item()) ** 0.5 if n_val > 0 else float('inf')

    if step % 10 == 0:
        print(f"Step {step:3d} | avg pred error: {aver_prediction_error:.4f} "
              f"| val RMSE: {val_rmse:.4f}")

    if val_rmse < best_val_rmse:
        best_val_rmse    = val_rmse
        best_movie_state = movie_features.data.clone()
        best_user_states = [uf.data.clone() for uf in user_features]
        patience_count   = 0
    else:
        patience_count += 1
        if patience_count >= patience:
            print(f"Early stopping at step {step}.")
            break

# Restore best checkpoint
movie_features.data.copy_(best_movie_state)
for uf, state in zip(user_features, best_user_states):
    uf.data.copy_(state)

print(f"\nBest val RMSE: {best_val_rmse:.4f}")


Step   0 | avg pred error: 64.8793 | val RMSE: 12.4821
Step  10 | avg pred error: 16.2188 | val RMSE: 5.7399
Step  20 | avg pred error: 3.0385 | val RMSE: 3.8838
Early stopping at step 26.

Best val RMSE: 2.7442


In [11]:
# test_rating_matrix is already built in Cell 2.
print(f"Test matrix shape : {test_rating_matrix.shape}")
print(f"Test observed     : {(test_rating_matrix != -1).sum().item()}")


Test matrix shape : torch.Size([1760, 2881])
Test observed     : 19450


In [12]:
# Item IDs are already remapped to contiguous 0-based integers,
# so test items map directly to the same column indices as train.
# The test_indices_in_train alias is set to identity for compatibility.
test_indices_in_train = list(range(n_items))
print("Item indices already aligned — no remapping needed.")


Item indices already aligned — no remapping needed.


In [13]:
print(movie_features)

tensor([[ 0.4640,  0.2500, -0.4509,  ...,  0.2725,  0.3067, -0.4720],
        [ 0.5053,  0.1127, -0.4608,  ...,  0.4246,  0.2710, -0.5055],
        [ 0.1353,  0.0976, -0.1194,  ...,  0.1455,  0.0964, -0.1516],
        ...,
        [ 0.0956,  0.0610, -0.0764,  ...,  0.0708,  0.0483, -0.0807],
        [ 0.1296,  0.0523, -0.1225,  ...,  0.0837,  0.0979, -0.1297],
        [ 0.1089,  0.0905, -0.1093,  ...,  0.0537,  0.0797, -0.0900]],
       requires_grad=True)


In [14]:
non_zero_mask = (test_rating_matrix[:num_client * m] != -1).type(torch.FloatTensor)
num           = torch.sum(non_zero_mask)

def Error(matrix, u_features, v_features):
    predicted_ratings = torch.sigmoid(torch.mm(u_features, v_features.t()))
    # test_indices_in_train is identity (0-based aligned), so no column reindex needed
    pred   = (predicted_ratings * (max_rating - min_rating) + min_rating) * non_zero_mask[i*m:(i+1)*m]
    actual = matrix * non_zero_mask[i*m:(i+1)*m]
    AE_diff = torch.abs(pred - actual)
    SE_diff = (pred - actual) ** 2
    return torch.sum(AE_diff), torch.sum(SE_diff)

AE_error = 0
SE_error = 0
for i in range(num_client):
    ae, se = Error(test_rating_matrix[i*m:(i+1)*m], user_features[i], movie_features)
    AE_error += ae
    SE_error += se

test_MAE  = AE_error / num
test_RMSE = torch.sqrt(SE_error / num)
print(f"Test MAE  : {test_MAE.item():.4f}")
print(f"Test RMSE : {test_RMSE.item():.4f}")


Test MAE  : 2.1253
Test RMSE : 2.7248


In [15]:
import numpy as np
def topk_metrics(y_true, y_pred, topKs=[10]):
	"""choice topk metrics and compute it
	the metrics contains 'ndcg', 'mrr', 'recall' and 'hit'

	Args:
		y_true: list, 2-dim, each row contains the items that the user interacted
		y_pred: list, 2-dim, each row contains the items recommended  
		topKs: list or tuple, if you want to get top5 and top10, topKs=(5, 10)

	Return:
		results: list, it contains five metrics, 'ndcg', 'recall', 'mrr', 'hit', 'precision'

	"""
	assert len(y_true) == len(y_pred)

	if not isinstance(topKs, (tuple, list)):
		raise ValueError('topKs wrong, it should be tuple or list')

	ndcg_result = []
	mrr_result = []
	hit_result = []
	precision_result = []
	recall_result = []
	for idx in range(len(topKs)):
		ndcgs = 0
		mrrs = 0
		hits = 0
		precisions = 0
		recalls = 0
		for i in range(len(y_true)):
			if len(y_true[i]) != 0:
				mrr_tmp = 0
				mrr_flag = True
				hit_tmp = 0
				dcg_tmp = 0
				idcg_tmp = 0
				hit = 0
				for j in range(topKs[idx]):
					if y_pred[i][j] in y_true[i]:
						hit += 1.
						if mrr_flag:
							mrr_flag = False
							mrr_tmp = 1. / (1 + j)
							hit_tmp = 1.
						dcg_tmp += 1. / (np.log2(j + 2))
					idcg_tmp += 1. / (np.log2(j + 2))
				hits += hit_tmp
				mrrs += mrr_tmp
				recalls += hit / len(y_true[i])
				precisions += hit / topKs[idx]
				if idcg_tmp != 0:
					ndcgs += dcg_tmp / idcg_tmp
		hit_result.append(round(hits / len(y_pred), 4))
		mrr_result.append(round(mrrs / len(y_pred), 4))
		recall_result.append(round(recalls / len(y_pred), 4))
		precision_result.append(round(precisions / len(y_pred), 4))
		ndcg_result.append(round(ndcgs / len(y_pred), 4))

	results = []
	for idx in range(len(topKs)):
		results.append(hit_result[idx])
	return results

In [16]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

non_zero_mask = (test_rating_matrix[:num_client * m] != -1).type(torch.FloatTensor)
num           = torch.sum(non_zero_mask)
print(f"Test observed entries: {int(num.item())}")

def recommend(u_features, v_features, mask, k):
    scores = torch.sigmoid(torch.mm(u_features, v_features.t())).detach().numpy()
    return [sorted(range(len(s)), key=lambda j: s[j], reverse=True)[:k]
            for s in scores]

def ground_truth(actual):
    return [[j for j, e in enumerate(row) if e != 0] for row in actual]


Test observed entries: 17645


In [17]:
Hit_Ratio_10 = 0
for i in range(num_client):
    actual_matrix = (test_rating_matrix[i*m:(i+1)*m] * non_zero_mask[i*m:(i+1)*m]).numpy()
    actual  = ground_truth(actual_matrix)
    predict = recommend(user_features[i], movie_features, non_zero_mask[i*m:(i+1)*m], 10)
    results = topk_metrics(actual, predict, topKs=(5, 10))
    Hit_Ratio_10 += results[1] / num_client
print(f"HR@10: {Hit_Ratio_10:.4f}")


HR@10: 0.0050


In [18]:
def score_f(u_features, v_features, mask, k):
    scores = (torch.sigmoid(torch.mm(u_features,v_features.t()))).data.numpy()
    # scores = (torch.sigmoid(torch.mm(u_features,v_features.t()))).data.numpy()
    recommend_list = []
    score_list = []
    # print(scores)
    for i in range(len(scores)):
    # print(torch.sum(mask))
    # print(scores.shape)
        score = scores[i]
        sorted_id = sorted(range(len(score)), key=lambda k: score[k], reverse=True)[:k]
        sorted_score = [score[j] for j in sorted_id]
        recommend_list.append(sorted_id)
        score_list.append(sorted_score)
    return recommend_list, score_list
def rel(actual_matrix, recommend):
    rel_list = []
    for i in range(len(actual_matrix)):
        actual_vector = actual_matrix[i]
        rel_id = [actual_vector[j] for j in recommend[i]]
        rel_list.append(rel_id)
    return rel_list

In [19]:
import numpy as np
from sklearn import metrics

ndcg_10 = 0
for i in range(num_client):
    actual_matrix  = (test_rating_matrix[i*m:(i+1)*m] * non_zero_mask[i*m:(i+1)*m]).numpy()
    recommend_list, score_list = score_f(
        user_features[i], movie_features, non_zero_mask[i*m:(i+1)*m], 10)
    true_relevance = rel(actual_matrix, recommend_list)
    ndcg_10 += metrics.ndcg_score(true_relevance, score_list, k=10) / num_client
print(f"NDCG@10: {ndcg_10:.4f}")


NDCG@10: 0.0026
